# 14. Recursive Forecasting

## Objective

The objective of this notebook is to generate multi-step sales forecasts using the optimized LightGBM models developed in the previous notebook.

Unlike one-step prediction, future lag and rolling features are unavailable during inference. Therefore, predictions are generated recursively by feeding previously predicted sales back into the feature engineering pipeline.

The notebook concludes by generating the final Kaggle competition submission.

## Workflow

1. Load engineered datasets
2. Train production LightGBM models
3. Validate the recursive forecasting pipeline
4. Generate recursive forecasts for the competition test period
5. Apply post-processing business rules
6. Create the final submission file

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Time series and statistics
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL

# Model
import joblib
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.optimize import minimize

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [8]:
PROCESSED_DATA = Path("../data/processed")

train = pd.read_csv(PROCESSED_DATA / "train.csv", parse_dates=["date"])
oil = pd.read_csv(PROCESSED_DATA / "oil.csv", parse_dates=["date"])
transactions = pd.read_csv(PROCESSED_DATA / "transactions.csv", parse_dates=["date"])

stores = pd.read_csv("../data/raw/stores.csv")
holidays = pd.read_csv("../data/raw/holidays_events.csv", parse_dates=["date"])
test = pd.read_csv("../data/raw/test.csv", parse_dates=["date"])

df = pd.read_csv(PROCESSED_DATA / "feature_matrix.csv", parse_dates=["date"])

train_fe = df[df["date"] <= "2017-08-15"].copy()
test_fe = df[df["date"] > "2017-08-15"].copy()

# 15. Train Production Models

In [3]:
# Final scheme: two models on recent periods
# model_2016 is trained on data from 2016-01-01
# model_2017 is trained on data from 2017-01-01

best_n = 5408
print(f"best_iteration_: 5408")

lgbm_params = dict(
    n_estimators=best_n,
    learning_rate=0.01,
    num_leaves=255,
    min_child_samples=30,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
FEATURES_V2 = [
    # calendar features
    "day_of_week", "month", "year", "is_weekend",
    "day_of_year", "week_of_year", "date_index",
    # Fourier seasonality features
    "sin_day", "cos_day", "sin_week", "cos_week",
    # lags: short-term 1-6, weekly/monthly, 6-8 weeks, annual
    "lag_1", "lag_2", "lag_3", "lag_4", "lag_5", "lag_6",
    "lag_7", "lag_14", "lag_21", "lag_28", "lag_42", "lag_56",
    "lag_364", "lag_365",
    # rolling means
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28", "rolling_mean_364",
    # oil and its moving averages
    "dcoilwtico", "dcoilwtico_ma7", "dcoilwtico_ma28",
    # promo and its moving averages
    "onpromotion", "onpromotion_ma7", "onpromotion_ma28",
    # transactions: lags 16-23 days, no leakage from test
    *[f"transactions_lag_{l}" for l in range(16, 24)],
    # holidays
    "is_holiday_national", "day_before_holiday",
    "is_black_friday", "is_cyber_monday", "is_terremoto",
    "is_navidad", "is_dia_madre", "is_futbol",
    "is_dia_trabajo", "is_primer_dia", "is_dia_difuntos", "work_day",
    # store
    "store_nbr", "store_type", "cluster",
    # product category
    "family",
]
TARGET = "sales"
CAT_FEATURES = ["store_nbr", "store_type", "cluster", "family", "day_of_week", "month"]
SPLIT_DATE = "2017-07-31"

# Data for model_2016
train_fe_2016 = train_fe[train_fe["date"] >= "2016-01-01"].copy()
X_2016 = train_fe_2016[FEATURES_V2].copy()
y_2016 = np.log1p(train_fe_2016["sales"])
for col in CAT_FEATURES:
    X_2016[col] = X_2016[col].astype("category")
model_2016 = lgb.LGBMRegressor(**lgbm_params)
model_2016.fit(X_2016, y_2016)
print("model_2016 trained:", X_2016.shape)

# Data for model_2017
train_fe_2017 = train_fe[train_fe["date"] >= "2017-01-01"].copy()
X_2017 = train_fe_2017[FEATURES_V2].copy()
y_2017 = np.log1p(train_fe_2017["sales"])
for col in CAT_FEATURES:
    X_2017[col] = X_2017[col].astype("category")
model_2017 = lgb.LGBMRegressor(**lgbm_params)
model_2017.fit(X_2017, y_2017)
print("model_2017 trained:", X_2017.shape)


best_iteration_: 5408
model_2016 trained: (1056726, 59)
model_2017 trained: (404514, 59)


# 15.1 Recursive Validation

Before generating forecasts for the competition test set, the recursive forecasting pipeline is validated on a historical period with known sales values.

Unlike one-step forecasting, recursive forecasting repeatedly feeds previous predictions back into the feature engineering process to generate future lag and rolling features. This validation step evaluates how well the complete forecasting pipeline performs under realistic inference conditions.

In [4]:


VAL_START = pd.Timestamp("2017-07-31")
VAL_END   = pd.Timestamp("2017-08-15")
val_dates = sorted(df[(df["date"] >= VAL_START) & (df["date"] <= VAL_END)]["date"].unique())

# Copy df and hide real sales for the validation period
df_val = df.copy()
df_val.loc[df_val["date"] >= VAL_START, "sales"] = 0.0

# Compute zero-rule only on data before validation
zero_val = set()
pre_val = df_val[df_val["date"] < VAL_START].groupby(["store_nbr", "family"]).apply(
    lambda x: x.nlargest(21, "date")["sales"].sum()
).reset_index()
pre_val.columns = ["store_nbr", "family", "last21_sum"]
for _, row in pre_val[pre_val["last21_sum"] == 0].iterrows():
    zero_val.add((row["store_nbr"], row["family"]))

# Compute rolling_mean_364 once over the long history
df_val["rolling_mean_364"] = (
    df_val.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(364, min_periods=30).mean())
)

# Iterative forecast: lags and rolling updated after each day
val_records = []
for pred_date in val_dates:
    for lag in [1, 2, 3, 4, 5, 6, 7, 14, 21, 28, 42, 56, 364, 365]:
        df_val[f"lag_{lag}"] = df_val.groupby(["store_nbr", "family"])["sales"].shift(lag)
    for w in [7, 14, 28]:
        df_val[f"rolling_mean_{w}"] = (
            df_val.groupby(["store_nbr", "family"])["sales"]
            .transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
        )
    mask = df_val["date"] == pred_date
    X_day = df_val.loc[mask, FEATURES_V2].copy()
    for col in CAT_FEATURES:
        X_day[col] = X_day[col].astype("category")

    p_2016 = np.expm1(model_2016.predict(X_day)).clip(0)
    p_2017 = np.expm1(model_2017.predict(X_day)).clip(0)

    # Write a neutral 50/50 blend to advance the loop
    day_pred = 0.5 * p_2016 + 0.5 * p_2017
    day_rows = df_val.loc[mask, ["store_nbr", "family"]].values
    for i, (s, f) in enumerate(day_rows):
        if (s, f) in zero_val:
            day_pred[i] = 0.0
    df_val.loc[mask, "sales"] = day_pred

    tmp = df_val.loc[mask, ["date", "store_nbr", "family"]].copy()
    tmp["p_2016"] = p_2016
    tmp["p_2017"] = p_2017
    val_records.append(tmp)
    print(f"{pred_date.date()}: done")

val_df = pd.concat(val_records)
true_sales = df[df["date"].between(VAL_START, VAL_END)][["date", "store_nbr", "family", "sales"]]
val_df = val_df.merge(true_sales, on=["date", "store_nbr", "family"])

def rmsle(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(y_pred.clip(0)) - np.log1p(y_true)) ** 2))

y_true = val_df["sales"].values
p16    = val_df["p_2016"].values
p17    = val_df["p_2017"].values

best_score = np.inf
best_w = (0.0, 1.0)
for w16 in np.arange(0.0, 1.01, 0.1):
    w17 = 1.0 - w16
    score = rmsle(y_true, w16 * p16 + w17 * p17)
    marker = " <-- best" if score < best_score else ""
    print(f"w_2016={w16:.1f}, w_2017={w17:.1f}  →  RMSLE={score:.5f}{marker}")
    if score < best_score:
        best_score = score
        best_w = (w16, w17)

w_2016, w_2017 = best_w
print(f"\nFair validation RMSLE: {best_score:.5f}")
print(f"w_2016={w_2016:.1f}, w_2017={w_2017:.1f}")
del df_val

2017-07-31: done
2017-08-01: done
2017-08-02: done
2017-08-03: done
2017-08-04: done
2017-08-05: done
2017-08-06: done
2017-08-07: done
2017-08-08: done
2017-08-09: done
2017-08-10: done
2017-08-11: done
2017-08-12: done
2017-08-13: done
2017-08-14: done
2017-08-15: done
w_2016=0.0, w_2017=1.0  →  RMSLE=0.27663 <-- best
w_2016=0.1, w_2017=0.9  →  RMSLE=0.27984
w_2016=0.2, w_2017=0.8  →  RMSLE=0.28346
w_2016=0.3, w_2017=0.7  →  RMSLE=0.28734
w_2016=0.4, w_2017=0.6  →  RMSLE=0.29139
w_2016=0.5, w_2017=0.5  →  RMSLE=0.29559
w_2016=0.6, w_2017=0.4  →  RMSLE=0.29990
w_2016=0.7, w_2017=0.3  →  RMSLE=0.30431
w_2016=0.8, w_2017=0.2  →  RMSLE=0.30882
w_2016=0.9, w_2017=0.1  →  RMSLE=0.31341
w_2016=1.0, w_2017=0.0  →  RMSLE=0.31808

Fair validation RMSLE: 0.27663
w_2016=0.0, w_2017=1.0


# 15.2 Competition Forecasting

After validating the recursive forecasting pipeline, the same recursive approach is applied to the competition test period.

Predictions are generated one day at a time, with previously predicted sales feeding back into the feature engineering process to create lag and rolling features for subsequent dates.

Finally, the submission file is created in the format required by the Kaggle competition.

In [10]:
# Zero-rule: if the last 21 train days were zero, forecast 0
import time; _t0 = time.time()
zero_series = set()
last_train = train.groupby(["store_nbr", "family"]).apply(
    lambda x: x.nlargest(21, "date")["sales"].sum()
).reset_index()
last_train.columns = ["store_nbr", "family", "last21_sum"]
for _, row in last_train[last_train["last21_sum"] == 0].iterrows():
    zero_series.add((row["store_nbr"], row["family"]))
print(f"Zero-rule: {len(zero_series)} series = 0")

# Compute rolling_mean_364 and external factor moving averages once
df["rolling_mean_364"] = (
    df.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(364, min_periods=30).mean())
)
for col in ["dcoilwtico", "onpromotion"]:
    for w in [7, 28]:
        df[f"{col}_ma{w}"] = (
            df.groupby(["store_nbr", "family"])[col]
            .transform(lambda x: x.rolling(w, min_periods=1).mean())
        )

# Iterative test forecast: lags and rolling updated from predictions
test_dates = sorted(test["date"].unique())
all_preds = []

for pred_date in test_dates:
    for lag in [1, 2, 3, 4, 5, 6, 7, 14, 21, 28, 42, 56, 364, 365]:
        df[f"lag_{lag}"] = df.groupby(["store_nbr", "family"])["sales"].shift(lag)
    for w in [7, 14, 28]:
        df[f"rolling_mean_{w}"] = (
            df.groupby(["store_nbr", "family"])["sales"]
            .transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
        )

    mask = df["date"] == pred_date
    X_day = df.loc[mask, FEATURES_V2].copy()
    for col in CAT_FEATURES:
        X_day[col] = X_day[col].astype("category")

    p_2016 = np.expm1(model_2016.predict(X_day)).clip(0)
    p_2017 = np.expm1(model_2017.predict(X_day)).clip(0)
    day_pred = w_2016 * p_2016 + w_2017 * p_2017

    day_rows = df.loc[mask, ["store_nbr", "family"]].values
    for i, (s, f) in enumerate(day_rows):
        if (s, f) in zero_series:
            day_pred[i] = 0.0

    df.loc[mask, "sales"] = day_pred
    tmp = df.loc[mask, ["date", "store_nbr", "family"]].copy()
    tmp["sales"] = day_pred
    all_preds.append(tmp)
    print(f"{pred_date.date()}: sum = {day_pred.sum():.0f}")

preds_df = pd.concat(all_preds)
submission = test.merge(
    preds_df[["date", "store_nbr", "family", "sales"]],
    on=["date", "store_nbr", "family"], how="left"
)[["id", "sales"]]
submission.to_csv("../outputs/submission/submission.csv", index=False)
print("Saved:", submission.shape)
print("NaN:", submission["sales"].isna().sum())
submission.head()
print(f"Execution time: {(time.time() - _t0) / 60:.1f} min")


Zero-rule: 127 series = 0
2017-08-16: sum = 789305
2017-08-17: sum = 643984
2017-08-18: sum = 729409
2017-08-19: sum = 823392
2017-08-20: sum = 921716
2017-08-21: sum = 756193
2017-08-22: sum = 707397
2017-08-23: sum = 714798
2017-08-24: sum = 624528
2017-08-25: sum = 731339
2017-08-26: sum = 828061
2017-08-27: sum = 930860
2017-08-28: sum = 772152
2017-08-29: sum = 743202
2017-08-30: sum = 786022
2017-08-31: sum = 692990
Saved: (28512, 2)
NaN: 0
Execution time: 1.2 min
